In [1]:
!pip install chromadb sentence-transformers langchain langchain-community langchain-text-splitters streamlit --quiet
print("✅ Libraries installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 

In [2]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✅ All imports done!")

/tmp/ipykernel_16/1694060332.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


✅ All imports done!


In [3]:
# Load embedding model
model = SentenceTransformer('all-mpnet-base-v2')
print("✅ Embedding model loaded!")
print(f"   Model dimension: {model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
   Model dimension: 768


/tmp/ipykernel_16/2740103679.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Model dimension: {model.get_sentence_embedding_dimension()}")


In [4]:
# CORRECT embedding function for ChromaDB
class EmbeddingFunction:
    def __init__(self, model):
        self.model = model
    
    def __call__(self, input):
        """For documents - ChromaDB requires 'input' parameter name"""
        if isinstance(input, str):
            input = [input]
        return self.model.encode(input).tolist()
    
    def embed_query(self, input):
        """For queries - parameter name MUST be 'input'"""
        if isinstance(input, str):
            input = [input]
        return self.model.encode(input).tolist()

print("✅ Embedding function defined!")

✅ Embedding function defined!


In [5]:
# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path='/kaggle/working/chroma_db')

# Delete old collection if exists
try:
    chroma_client.delete_collection("company_policies")
    print("✅ Deleted old collection")
except:
    print("ℹ️ No existing collection to delete")

# Create new collection
collection = chroma_client.create_collection(
    name="company_policies",
    embedding_function=EmbeddingFunction(model)
)

print("✅ ChromaDB collection created!")
print(f"   Collection name: {collection.name}")
print(f"   Initial count: {collection.count()}")

ℹ️ No existing collection to delete
✅ ChromaDB collection created!
   Collection name: company_policies
   Initial count: 0


In [6]:
# Create documents folder
os.makedirs('/kaggle/working/company_docs', exist_ok=True)

# Sample documents
documents = {
    "vacation_policy.txt": """
        Employee Vacation Policy
        
        All full-time employees receive 20 paid vacation days per year.
        Vacation days accrue monthly at a rate of 1.67 days per month.
        Unused vacation days expire after 1 year from the date of hire.
        Employees must request vacation at least 2 weeks in advance.
    """,
    
    "remote_work.txt": """
        Remote Work Policy
        
        Employees may work from home up to 3 days per week.
        Remote work requires manager approval.
        Employees working remotely must be available during core hours (10 AM - 3 PM).
        Internet stipend of $50 per month is provided for remote workers.
    """,
    
    "maternity_leave.txt": """
        Parental Leave Policy
        
        Employees receive 90 days (12 weeks) of paid maternity leave.
        Primary caregivers receive full pay during leave period.
        Secondary caregivers receive 4 weeks of paid leave.
        Parental leave can be taken any time within the first year after birth.
    """
}

# Write documents
for filename, content in documents.items():
    with open(f'/kaggle/working/company_docs/{filename}', 'w') as f:
        f.write(content.strip())

print("✅ Sample documents created!")
print("\n📁 Documents created:")
for filename in documents.keys():
    print(f"   📄 {filename}")

✅ Sample documents created!

📁 Documents created:
   📄 vacation_policy.txt
   📄 remote_work.txt
   📄 maternity_leave.txt


In [7]:
# Load documents
loader = DirectoryLoader(
    '/kaggle/working/company_docs/', 
    glob='*.txt', 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

docs = loader.load()
print(f"✅ Loaded {len(docs)} documents")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '.', ' ', '']
)

chunks = text_splitter.split_documents(docs)
print(f"✅ Split into {len(chunks)} chunks")

# Show chunks
print("\n📖 Chunk previews:")
for i, chunk in enumerate(chunks[:3]):
    preview = chunk.page_content.replace('\n', ' ')[:80]
    print(f"   Chunk {i+1}: {preview}...")

✅ Loaded 3 documents
✅ Split into 3 chunks

📖 Chunk previews:
   Chunk 1: Employee Vacation Policy                  All full-time employees receive 20 pai...
   Chunk 2: Parental Leave Policy                  Employees receive 90 days (12 weeks) of p...
   Chunk 3: Remote Work Policy                  Employees may work from home up to 3 days pe...


In [8]:
# Prepare for indexing
chunk_texts = [chunk.page_content for chunk in chunks]
chunk_ids = [f'doc_{i}' for i in range(len(chunks))]

# Add to ChromaDB
collection.add(
    documents=chunk_texts,
    ids=chunk_ids
)

print(f"✅ Added {len(chunks)} documents to ChromaDB")
print(f"📊 Total in collection: {collection.count()}")

# Verify
print("\n📁 Indexed documents:")
for i, text in enumerate(chunk_texts):
    preview = text.replace('\n', ' ')[:60]
    print(f"   ID {chunk_ids[i]}: {preview}...")

✅ Added 3 documents to ChromaDB
📊 Total in collection: 3

📁 Indexed documents:
   ID doc_0: Employee Vacation Policy                  All full-time empl...
   ID doc_1: Parental Leave Policy                  Employees receive 90 ...
   ID doc_2: Remote Work Policy                  Employees may work from ...


In [9]:
def semantic_search(query, n_results=2):
    """Search using semantic similarity"""
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

def generate_response(query):
    """Generate response based on retrieved documents"""
    results = semantic_search(query, n_results=1)
    
    if not results['documents'][0]:
        return "❌ No relevant information found in company documents."
    
    doc = results['documents'][0][0]
    distance = results['distances'][0][0]
    similarity = 1 / (1 + distance)  # Convert distance to similarity
    
    # Create response
    response = f"{doc}\n\n📊 *Relevance: {similarity:.1%}*"
    
    # Add source label
    if 'vacation' in doc.lower():
        response += "\n📁 *Source: Vacation Policy*"
    elif 'remote' in doc.lower() or 'work from home' in doc.lower():
        response += "\n📁 *Source: Remote Work Policy*"
    elif 'maternity' in doc.lower() or 'parental' in doc.lower():
        response += "\n📁 *Source: Parental Leave Policy*"
    
    return response

print("✅ Search functions ready!")
print("\n📊 Test the search:")
test = semantic_search("vacation days")
print(f"   Query 'vacation days' found: {len(test['documents'][0])} results")

✅ Search functions ready!

📊 Test the search:
   Query 'vacation days' found: 2 results


In [10]:
print("="*60)
print("🔍 TESTING SEMANTIC SEARCH")
print("="*60)

test_queries = [
    ("time off", "Should find vacation policy"),
    ("work from home", "Should find remote work"),
    ("baby leave", "Should find maternity leave"),
    ("dress code", "Should find nothing or low relevance")
]

for query, expected in test_queries:
    print(f"\n❓ Query: '{query}'")
    print(f"   Expected: {expected}")
    print("-"*40)
    
    results = semantic_search(query, n_results=1)
    
    if results['documents'][0]:
        doc = results['documents'][0][0]
        distance = results['distances'][0][0]
        similarity = 1 / (1 + distance)
        
        # Identify which document
        if 'vacation' in doc.lower():
            source = "📁 Vacation Policy"
        elif 'remote' in doc.lower():
            source = "📁 Remote Work Policy"
        elif 'maternity' in doc.lower() or 'parental' in doc.lower():
            source = "📁 Parental Leave Policy"
        else:
            source = "📁 Unknown"
        
        print(f"   ✅ Found: {doc[:100]}...")
        print(f"   📊 Similarity: {similarity:.2%}")
        print(f"   📁 Source: {source}")
    else:
        print("   ❌ No results found")
    print("-"*40)

print("\n✨ SUCCESS! Semantic search finds documents by MEANING!")

🔍 TESTING SEMANTIC SEARCH

❓ Query: 'time off'
   Expected: Should find vacation policy
----------------------------------------
   ✅ Found: Employee Vacation Policy
        
        All full-time employees receive 20 paid vacation days per ...
   📊 Similarity: 40.22%
   📁 Source: 📁 Vacation Policy
----------------------------------------

❓ Query: 'work from home'
   Expected: Should find remote work
----------------------------------------
   ✅ Found: Remote Work Policy
        
        Employees may work from home up to 3 days per week.
        Remo...
   📊 Similarity: 51.26%
   📁 Source: 📁 Remote Work Policy
----------------------------------------

❓ Query: 'baby leave'
   Expected: Should find maternity leave
----------------------------------------
   ✅ Found: Parental Leave Policy
        
        Employees receive 90 days (12 weeks) of paid maternity leave....
   📊 Similarity: 42.51%
   📁 Source: 📁 Parental Leave Policy
----------------------------------------

❓ Query: 'dress